In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
words = open('names.txt', 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [7]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0 # add a special token for the end of a word
itos = {i:ch for ch, i in stoi.items()}
print(stoi)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [63]:
box_size = 3
X, Y = [], []
for w in words:
    context = [0] * box_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # think of it like a rolling window of context

X = torch.tensor(X)
Y = torch.tensor(Y)

In [22]:
X.shape

torch.Size([32, 3])

In [41]:
X[:10]

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22]])

In [64]:
C = torch.randn((27, 2)) # this is the character representation matrix, which will be learned. 27 rows for each character, 2 columns for the embedding dimension
C.shape

torch.Size([27, 2])

In [26]:
emb = C[X] # (num_samples, context_size, embedding_dim)
emb.shape

torch.Size([32, 3, 2])

In [27]:
# creating the hidden layer
W1 = torch.randn((6, 100)) # 6 = context_size * embedding_dim, 100 = hidden layer size
b1 = torch.randn(100) # bias for the hidden layer

In [39]:
torch.cat(torch.unbind(emb, 1), dim=1).shape

torch.Size([32, 6])

In [44]:
# another way of doing this using view
emb.view(-1, 6) # we use -1 because -1 can be inferred from the other dimensions, so we don't have to calculate it manually
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (num_samples, hidden_layer_size)
h.shape

torch.Size([32, 100])

In [45]:
W2 = torch.randn((100, 27)) # 100 = hidden layer size, 27 = output size (number of characters)
b2 = torch.randn(27) # bias for the output layer

In [46]:
logits = h @ W2 + b2 # (num_samples, output_size)
logits.shape

torch.Size([32, 27])

In [50]:
count = logits.exp()
probs = count / count.sum(1, keepdim=True) # normalize for each row
probs

tensor([[4.1826e-05, 6.7835e-04, 1.2658e-10, 1.4014e-06, 3.0769e-01, 5.1873e-12,
         1.6796e-09, 1.3599e-07, 1.4062e-05, 6.6781e-08, 2.2680e-01, 1.2609e-06,
         1.4779e-05, 3.1842e-08, 4.3145e-05, 6.9344e-08, 5.7239e-14, 2.2258e-09,
         5.3243e-07, 1.1038e-11, 2.6816e-05, 4.6467e-01, 8.0807e-10, 6.3938e-13,
         8.9461e-12, 5.1662e-13, 2.6222e-05],
        [2.1665e-04, 6.4212e-04, 3.4000e-11, 1.2551e-06, 6.4716e-01, 1.6244e-12,
         2.4625e-09, 1.0366e-07, 1.1906e-05, 9.9387e-08, 1.1612e-01, 3.6979e-07,
         1.1081e-05, 4.0313e-08, 1.1682e-04, 7.3367e-07, 3.5281e-13, 4.4262e-09,
         4.4126e-07, 1.2818e-11, 3.0453e-05, 2.3568e-01, 1.4875e-10, 2.6631e-13,
         1.0242e-11, 2.8552e-13, 1.0218e-05],
        [6.4224e-11, 3.7532e-02, 2.3469e-07, 4.3114e-05, 2.5079e-04, 2.1002e-05,
         1.3202e-06, 6.3417e-04, 3.0979e-08, 2.3801e-07, 3.3067e-04, 1.8409e-06,
         1.5166e-08, 2.1667e-08, 7.6174e-01, 6.5912e-07, 4.1266e-13, 1.5659e-09,
         1.6507e-

In [136]:
# a way of doing it even faster
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g) # this is the character representation matrix, which will be learned. 27 rows for each character, 2 columns for the embedding dimension
W1 = torch.randn((6, 100), generator=g) # 6 = context_size * embedding_dim, 100 = hidden layer size
b1 = torch.randn(100, generator=g) # bias for the hidden layer
W2 = torch.randn((100, 27), generator=g) # 100 = hidden layer size, 27 = output size (number of characters)
b2 = torch.randn(27, generator=g) # bias for the output layer
parameters = [C, W1, b1, W2, b2]

In [137]:
sum(p.numel() for p in parameters) # number of parameters in the model

3481

In [138]:
# emb = C[X] # (num_samples, context_size, embedding_dim)
# h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (num_samples, hidden_layer_size)
# logits = h @ W2 + b2 # (num_samples, output_size)
# counts = logits.exp()
# probs = counts / counts.sum(1, keepdim=True) # normalize for each row
# loss = -probs[torch.arange(probs.shape[0]), Y].log().mean() # negative log likelihood loss
# loss

In [139]:
for p in parameters:
    p.requires_grad = True

In [140]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre
# we are using the exponential function to get the learning rates because we want to explore a wide range of learning rates, from very small to very large. The exponential function allows us to do this in a smooth way.

In [145]:
# this could be done in an easier way using the cross entropy loss function
lri = []
lossi = []

for i in range(100000):
    ix = torch.randint(0, X.shape[0], (32,), generator=g) # we add this so that we can do it in batches, 32 random integers between 0 and X.shape[0] - 1
    # forward pass
    emb = C[X[ix]] # (num_samples, context_size, embedding_dim)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (num_samples, hidden_layer_size)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    # backward pass
    for p in parameters:
        p.grad = None # set the gradients to zero before running the backward pass
    loss.backward() # compute the gradients
    # print(loss.item())
    lr = 0.1
    #lr = lrs[i] # learning rate
    for p in parameters:
        p.data += -lr * p.grad # update the parameters using gradient descent
    # lri.append(lre[i].item())
    # lossi.append(loss.item())
print(loss.item())

2.0187442302703857


In [143]:
# plt.plot(lri, lossi)
# plt.xlabel('Learning Rate')
# plt.ylabel('Loss')
# plt.title('Loss vs Learning Rate')
# plt.legend()
# plt.show()


In [147]:
emb = C[X] # (num_samples, context_size, embedding_dim)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (num_samples, hidden_layer_size)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(2.2540, grad_fn=<NllLossBackward0>)